# Kumatage over a sea with a wind gradient

The [kumatage tutorial](../kumatage.ipynb) assumes the same wind everywhere. A
coastal observer often sees something else: close to the shore the sea is
sheltered (a light offshore wind, short fetch), while farther out the wind
blows freely and the sea is rough. This notebook simulates the glitter path
seen by an observer on a cliff or a ship deck when the wind speed increases
with the distance: **2 m/s near the observer and 12 m/s farther out**.

Two questions guide the study:

* how is a wind front a few kilometers away mapped into the field of view of
  the observer?
* what is its signature on the kumatage: brightness, width, polarization?

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm

from coxmunk import sunglint

## Geometry on a curved sea

The observer stands at the height $h$ above the sea and looks with a depression
angle $\delta$ below the horizontal and an azimuth offset $\alpha$ from the Sun.
The line of sight reaches the sea at the distance $d$. Since the waves farther
out are on a curved surface, the local vertical of the observed point is tilted
by the central angle $\gamma = d / R$ with respect to the vertical of the
observer, away from the observer. The light ray refraction is accounted for with
the usual effective Earth radius $R = R_\oplus / (1 - k)$, $k = 0.13$.

Writing $a = R + h$, the line of sight hits the sea at the range

$$t = a\sin\delta - \sqrt{a^2\sin^2\delta - (2Rh + h^2)},
\qquad \tan\gamma = \frac{t\cos\delta}{a - t\sin\delta},$$

and does not reach it above the dip of the horizon, $\cos\delta_h = R / (R + h)$.

In [ ]:
R = 6371e3 / (1 - 0.13)   # effective Earth radius with standard refraction (m)
h = 20.                    # height of the observer above the sea (m)


def sea_point(delta, h=h):
    """Distance (m) and central angle (deg) of the sea point seen at the depression delta (deg).
    NaN above the horizon."""
    d = np.radians(delta)
    a = R + h
    disc = (a * np.sin(d)) ** 2 - (2 * R * h + h ** 2)
    t = np.where(disc >= 0, a * np.sin(d) - np.sqrt(np.clip(disc, 0, None)), np.nan)
    gamma = np.arctan2(t * np.cos(d), a - t * np.sin(d))
    return R * gamma, np.degrees(gamma)


dip = np.degrees(np.arccos(R / (R + h)))
print(f'observer at {h:.0f} m: dip of the horizon {dip:.3f} deg, horizon at {sea_point(dip * 1.0001)[0] / 1e3:.1f} km')

In the frame of the observer, the observed point sees the observer at the
zenith angle $\theta_v = 90° - \delta$ and the azimuth $\phi = 180° + \alpha$
(Sun at azimuth 0), exactly as in the kumatage tutorial. The curvature is
handled by the tilted reference plane of {class}`coxmunk.coxmunk.sunglint`:
the local mean sea surface has the slope $\gamma$ and its normal leans toward
the azimuth $\alpha$ (away from the observer). With the default
`pol_ref='global'`, the Stokes vector stays referenced to the vertical plane of
the observer, i.e., what the observer would measure.

The local viewing zenith angle must be $90° - (\delta - \gamma)$: it is grazing
at the horizon, where $\delta = \gamma$.

In [ ]:
sun_elevation = 10.        # deg above the horizon
sza = 90. - sun_elevation


def glint(delta, alpha, ws, wazi=0, **kwargs):
    """Glint Stokes vector [I, Q, U, V] seen by the observer at (delta, alpha)."""
    gamma = sea_point(delta)[1]
    if np.isnan(gamma):
        return [np.nan] * 4
    return sunglint(sza, 90. - delta, 180. + alpha, tilt=gamma, tilt_azi=alpha).sunglint(
        ws, wazi, stats='bh2006', **kwargs)


g = sunglint(sza, 90. - 0.2, 180. + 5, tilt=sea_point(0.2)[1], tilt_azi=5)
print(f'delta = 0.2 deg: local viewing zenith angle {np.degrees(g.vza):.3f} deg, '
      f'expected {90 - (0.2 - sea_point(0.2)[1]):.3f} deg')

## The wind field

The wind blows offshore, away from the observer and toward the Sun
(`wazi=0`). Its speed rises from 2 m/s near the shore to 12 m/s beyond a wind
front at $d_f = 2$ km, over a transition length $L = 300$ m:

$$U(d) = U_{near} + \frac{U_{far} - U_{near}}{1 + e^{-(d - d_f)/L}}.$$

In [ ]:
ws_near, ws_far = 2., 12.
d_front, L_front = 2000., 300.


def wind(d):
    return ws_near + (ws_far - ws_near) / (1 + np.exp(-(d - d_front) / L_front))


delta = np.geomspace(dip * 1.02, 30, 140)     # depression below the horizontal (deg)
alpha = np.linspace(-30, 30, 121)             # azimuth offset from the Sun (deg)
dist = sea_point(delta)[0]
delta_front = np.interp(d_front, dist[::-1], delta[::-1])

fig, axs = plt.subplots(1, 2, figsize=(14, 4.5))
d_plot = np.linspace(0, 16e3, 400)
axs[0].plot(d_plot / 1e3, wind(d_plot))
axs[0].set_xlabel('distance from the observer (km)')
axs[0].set_ylabel('wind speed (m/s)')
axs[1].semilogx(delta, dist / 1e3)
axs[1].axvline(delta_front, color='k', ls=':', label=f'wind front: {delta_front:.2f} deg')
axs[1].axvline(sun_elevation, color='goldenrod', ls='--', label='flat-sea image of the Sun')
axs[1].set_xlabel('depression angle below the horizontal (deg)')
axs[1].set_ylabel('distance (km)')
axs[1].legend()
plt.tight_layout()

The whole rough sea, from 2 km to the horizon at 17 km, is squeezed into a
band about 0.45° high just below the horizon, less than the apparent diameter of
the Sun (0.53°). The calm sea fills the rest of the view, including the
flat-sea image of the Sun at $\delta = 10°$, only 110 m away. In the following,
the depression angle is plotted on a logarithmic axis to resolve this band.

## Glitter path: uniform winds and wind gradient

In [ ]:
def view(wind_of_distance):
    S = np.full((4, len(delta), len(alpha)), np.nan)
    for i, (dl, d) in enumerate(zip(delta, dist)):
        ws = wind_of_distance(d)
        for j, a in enumerate(alpha):
            S[:, i, j] = glint(dl, a, ws)
    return S


cases = {
    f'uniform {ws_near:.0f} m/s': lambda d: ws_near,
    f'uniform {ws_far:.0f} m/s': lambda d: ws_far,
    f'{ws_near:.0f} m/s near, {ws_far:.0f} m/s beyond {d_front / 1e3:.0f} km': wind,
}
stokes = {name: view(f) for name, f in cases.items()}


def deck_axes(ax, title):
    ax.set_yscale('log')
    ax.set_ylim(30, dip)
    ax.axhline(delta_front, color='w', ls=':', lw=1)
    ax.set_facecolor('#4a6fa5')
    ax.set_title(title)
    ax.set_xlabel('azimuth from the Sun (deg)')


fig, axs = plt.subplots(1, 3, figsize=(18, 6), sharey=True)
for ax, (name, S) in zip(axs, stokes.items()):
    im = ax.pcolormesh(alpha, delta, np.clip(S[0], 1e-6, None), norm=LogNorm(1e-2, 1e4),
                       cmap='inferno', shading='auto')
    deck_axes(ax, name)
axs[0].set_ylabel('depression below the horizontal (deg, log scale)')
fig.colorbar(im, ax=axs, label='glint reflectance I', shrink=0.9)
plt.show()

The dotted line marks the wind front. With the gradient, the view is that of
the calm sea up to the front, then switches to that of the rough sea in the
last half degree below the horizon: the glitter path, which narrows toward
the horizon over the calm sea, **widens again** beyond the front and its core
becomes **dimmer**.

## Brightness below the Sun

Below the Sun ($\alpha = 0$), the rough sea is *darker* than the calm sea: the
facets needed to reflect the Sun toward the observer are only slightly tilted,
and the rougher the sea, the smaller their proportion. The kumatage over the
distant rough water is therefore dimmer in its center than the glitter of a calm
sea would be.

The Sun is not a point, and the rough sea occupies a band thinner than the
solar disk. The profile is therefore also computed for the limb-darkened solar
disk (radius 0.2665°, 19 points), with the method of
[the kumatage with the apparent size of the Sun](kumatage_sun_disk.ipynb).

In [ ]:
# limb-darkened solar disk, as in the kumatage_sun_disk notebook
R_SUN = 0.2665           # apparent radius of the Sun (deg)


def disk_samples(n_rings=2, radius=R_SUN, u=0.6):
    """Offsets (d_elevation, d_horizontal) in deg of points of the solar disk and their weights."""
    pts = [(0., 0.)]
    for k in range(1, n_rings + 1):
        r, n = radius * k / (n_rings + 0.5), 6 * k
        pts += [(r * np.cos(2 * np.pi * (m + 0.5 * (k % 2)) / n), r * np.sin(2 * np.pi * (m + 0.5 * (k % 2)) / n))
                for m in range(n)]
    pts = np.array(pts)
    mu = np.sqrt(np.clip(1 - (np.hypot(*pts.T) / radius) ** 2, 0, 1))
    w = 1 - u * (1 - mu)
    return pts, w / w.sum()


def sun_angles(sun_elevation, offsets):
    """Zenith angles and azimuths (deg) of the points of the disk; Sun center at azimuth 0."""
    e0 = np.radians(sun_elevation)
    s0 = np.array([np.cos(e0), 0., np.sin(e0)])
    e_up = np.array([-np.sin(e0), 0., np.cos(e0)])
    e_h = np.array([0., 1., 0.])
    s = s0 + np.tan(np.radians(offsets[:, :1])) * e_up + np.tan(np.radians(offsets[:, 1:])) * e_h
    s /= np.linalg.norm(s, axis=1, keepdims=True)
    return np.degrees(np.arccos(s[:, 2])), np.degrees(np.arctan2(s[:, 1], s[:, 0]))


def profile(wind_of_distance, alpha=0., disk=False):
    """Glint I seen at the azimuth alpha from the Sun, for all depressions; point Sun or solar disk.
    The sensor, the wind and the slope of the sea keep their directions when the Sun moves."""
    off, w = disk_samples() if disk else (np.zeros((1, 2)), np.ones(1))
    I = np.zeros(len(delta))
    for sza_j, phi_j, w_j in zip(*sun_angles(sun_elevation, off), w):
        for i, (dl, d) in enumerate(zip(delta, dist)):
            g = sunglint(sza_j, 90. - dl, 180. + alpha - phi_j, tilt=sea_point(dl)[1], tilt_azi=alpha - phi_j)
            I[i] += w_j * g.sunglint(wind_of_distance(d), -phi_j, stats='bh2006')[0] \
                * np.cos(np.radians(sza_j)) / np.cos(np.radians(sza))
    return I


fig, ax = plt.subplots(figsize=(10, 5))
for (name, f), color in zip(cases.items(), ('C0', 'C3', 'k')):
    ax.loglog(delta, profile(f), color=color, label=name)
ax.loglog(delta, profile(wind, disk=True), 'k--', label='gradient, solar disk')
ax.axvline(delta_front, color='0.5', ls=':')
ax.axvline(sun_elevation, color='goldenrod', ls='--')
ax.set_xlim(dip, 30)
ax.invert_xaxis()
ax.set_xlabel('depression below the horizontal (deg) - horizon on the right')
ax.set_ylabel('glint reflectance I')
ax.set_title(r'Below the Sun ($\alpha = 0$)')
sec = ax.secondary_xaxis('top', functions=(lambda x: np.interp(x, delta, dist / 1e3),
                                           lambda x: np.interp(x, dist[::-1] / 1e3, delta[::-1])))
sec.set_xlabel('distance (km)')
sec.set_xticks([0.1, 0.5, 1, 2, 5, 10])
ax.legend()
plt.show()

Crossing the front, the reflectance below the Sun drops by a factor of about
5, from the calm-sea to the rough-sea level, over a few tenths of a degree.
This dimming is the strongest signature of the wind front. The finite solar
disk does not blur it: the transition, spread over about 1.5 km, still spans
more than the disk.

## Width of the glitter path

The width of the glitter path is measured, at each depression angle, by the
range of azimuths where the glint is brighter than a white Lambertian surface
($I > 1$).

In [ ]:
def path_width(I, threshold=1.):
    bright = I > threshold
    hi = np.where(bright, alpha, -np.inf).max(axis=1)
    lo = np.where(bright, alpha, np.inf).min(axis=1)
    return np.where(bright.any(axis=1), hi - lo, 0.)


fig, ax = plt.subplots(figsize=(10, 5))
for (name, S), color in zip(stokes.items(), ('C0', 'C3', 'k')):
    ax.semilogx(delta, path_width(S[0]), color=color, label=name)
ax.axvline(delta_front, color='0.5', ls=':', label='wind front')
ax.set_xlim(dip, 30)
ax.invert_xaxis()
ax.set_xlabel('depression below the horizontal (deg) - horizon on the right')
ax.set_ylabel('width of the glitter path (deg of azimuth)')
ax.legend()
plt.show()

The width follows the calm-sea curve up to the front, then joins the
rough-sea curve: about 11° of azimuth at the horizon instead of 6° for a
uniformly calm sea (the steps come from the 0.5° azimuth sampling). Together
with the dimming of its core, a glitter path that widens again near the
horizon reveals a rougher sea in the distance; the depression angle of the
transition gives the distance of the wind front, once the height of the
observer is known.

## Polarization

The degree and the angle of polarization only depend on the geometry of the
reflection, not on the wind (see the [polarization tutorial](../polarization.ipynb)).
The wind gradient changes where the polarized glint is seen, not its
polarization: wherever the glint is visible for both winds, DoLP and AoP are the same.

In [ ]:
I, Q, U, V = stokes[list(cases)[2]]
mask = I > 1e-2
dolp = np.where(mask, np.hypot(Q, U) / I, np.nan)
aolp = np.where(mask, np.degrees(0.5 * np.arctan2(U, Q)), np.nan)

fig, axs = plt.subplots(1, 2, figsize=(14, 6), sharey=True)
im = axs[0].pcolormesh(alpha, delta, dolp, vmin=0, vmax=1, cmap='viridis', shading='auto')
fig.colorbar(im, ax=axs[0], label='degree of linear polarization')
im = axs[1].pcolormesh(alpha, delta, aolp, vmin=-90, vmax=90, cmap='twilight', shading='auto')
cb = fig.colorbar(im, ax=axs[1], label=r'angle of polarization $\chi$ (deg)')
cb.set_ticks([-90, -45, 0, 45, 90])
for ax in axs:
    deck_axes(ax, 'wind gradient')
    ax.set_facecolor('0.6')
axs[0].set_ylabel('depression below the horizontal (deg, log scale)')
plt.show()

both = (stokes[list(cases)[0]][0] > 1e-2) & (stokes[list(cases)[1]][0] > 1e-2)
dolp_near, dolp_far = (np.hypot(S[1], S[2]) / S[0] for S in list(stokes.values())[:2])
print('max |DoLP(2 m/s) - DoLP(12 m/s)| where both glints are visible:',
      np.nanmax(np.abs(dolp_near - dolp_far)[both]))

## Effect of the Earth curvature

Neglecting the curvature (horizontal reference plane everywhere) changes the
local viewing angle by $\gamma$, at most the dip of the horizon (0.13° here),
for the most distant sea:

In [ ]:
flat = np.array([sunglint(sza, 90. - dl, 180.).sunglint(wind(d), 0, stats='bh2006')[0]
                 for dl, d in zip(delta, dist)])
curved = profile(wind)

fig, ax = plt.subplots(figsize=(10, 4))
ax.semilogx(delta, 100 * (flat / curved - 1))
ax.axvline(delta_front, color='0.5', ls=':')
ax.set_xlim(dip, 30)
ax.invert_xaxis()
ax.set_xlabel('depression below the horizontal (deg) - horizon on the right')
ax.set_ylabel('flat / curved - 1 (%)')
ax.set_title(r'Below the Sun, wind gradient')
plt.show()

The error stays below 0.2 %, even at the horizon: for a 20 m high observer,
the curvature of the Earth does not change the glint, only the distance at
which each depression angle meets the sea.

## Limitations

* The wind is assumed uniform along each line of sight footprint and the sea
  state is in equilibrium with the local wind: in reality the waves need some
  fetch to develop after the front, which spreads the transition.
* Only the glint is modelled: no sky reflection, whitecaps (frequent at
  12 m/s), water-leaving radiance nor atmospheric attenuation, which is strong
  along the near-horizontal paths toward the horizon.
* The facet slopes are sampled with the Cox-Munk statistics, valid for the
  slopes of the facets but not for the wave heights: the masking of the far sea
  by the swell crests is not described beyond the shadowing function.